# 🌾 Rice Leaf Disease Classification
## Deep Learning Architectures Showdown

**Author:** Md. Rakib Ahmed  
**University:** East West University  
**Program:** MSc in Data Science and Analytics  
**Supervisor:** Dr. Mohammad Rezaul Karim

---

### 📋 What This Notebook Does (Step by Step)

| Step | Description |
|------|-------------|
| 1 | Connect to Google Drive & install dependencies |
| 2 | Extract and explore the dataset |
| 3 | Remove duplicate images |
| 4 | Split data into Train / Validation / Test |
| 5 | Train VGG16, ResNet50, InceptionV3, Xception |
| 6 | K-Fold Cross Validation |
| 7 | Full evaluation (accuracy, F1, AUC, confusion matrix) |
| 8 | Statistical tests (McNemar, Wilcoxon, t-test) |
| 9 | Grad-CAM explainability visualizations |
| 10 | Save model + results back to Google Drive |

---
### ⚡ Before Running
**Runtime → Change runtime type → T4 GPU**  
This is free on Google Colab and makes training ~20x faster.


---
## STEP 1 — Setup: Connect Drive & Install Libraries

In [ ]:
# ============================================================
# Install required libraries
# (Most are pre-installed on Colab, but we pin versions)
# ============================================================
!pip install -q keras-tuner==1.3.5
!pip install -q scipy==1.11.4

print('✅ Libraries installed')

In [ ]:
# ============================================================
# Mount Google Drive
# This gives Colab access to your Drive files
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

print('✅ Google Drive mounted at /content/drive')

In [ ]:
# ============================================================
# CONFIGURATION — Change these paths to match YOUR Drive
# ============================================================

import os

# -----------------------------------------------------------
# ⚠️  UPDATE THESE TWO LINES TO MATCH YOUR GOOGLE DRIVE
# -----------------------------------------------------------
# Path to your .7z dataset file in Drive
DRIVE_DATA_PATH = '/content/drive/MyDrive/Thesis Data/Rice Leaf Disease Images.7z'

# Folder in Drive where we save all outputs (models, plots, reports)
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Thesis Data/outputs'
# -----------------------------------------------------------

# Local working paths inside Colab (fast SSD, temporary)
LOCAL_DATA_DIR    = '/content/Rice Leaf Disease Images'
LOCAL_SPLIT_DIR   = '/content/rice_leaf_split'
LOCAL_OUTPUT_DIR  = '/content/outputs'
MODEL_SAVE_DIR    = f'{LOCAL_OUTPUT_DIR}/models'
PLOTS_SAVE_DIR    = f'{LOCAL_OUTPUT_DIR}/plots'
REPORTS_SAVE_DIR  = f'{LOCAL_OUTPUT_DIR}/reports'

# Training settings
IMAGE_SIZE   = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 30
N_FOLDS      = 5
RANDOM_SEED  = 42
LEARNING_RATE = 0.0001

# Disease class names — must match your folder names exactly
CLASS_NAMES = ['Bacterialblight', 'Blast', 'Brownspot', 'Tungro']
NUM_CLASSES = len(CLASS_NAMES)

# Create local output directories
for d in [MODEL_SAVE_DIR, PLOTS_SAVE_DIR, REPORTS_SAVE_DIR, LOCAL_SPLIT_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Configuration set')
print(f'   Dataset path : {DRIVE_DATA_PATH}')
print(f'   Output dir   : {DRIVE_OUTPUT_DIR}')
print(f'   Classes      : {CLASS_NAMES}')

In [ ]:
# ============================================================
# Import all libraries
# ============================================================
import os
import json
import shutil
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.applications   import VGG16, ResNet50, InceptionV3, Xception
from tensorflow.keras.layers         import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models         import Model
from tensorflow.keras.optimizers     import Adam
from tensorflow.keras.callbacks      import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

# Scikit-learn
from sklearn.model_selection        import train_test_split, StratifiedKFold
from sklearn.utils.class_weight     import compute_class_weight
from sklearn.metrics                import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)
from sklearn.preprocessing          import label_binarize
from sklearn.metrics                import roc_curve, auc as sk_auc

# Statistical tests
from scipy                          import stats

# Reproducibility
import random
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

# Verify GPU
gpus = tf.config.list_physical_devices('GPU')
print(f'✅ Libraries imported')
print(f'   TensorFlow version : {tf.__version__}')
print(f'   GPU available      : {len(gpus) > 0}')
if gpus:
    print(f'   GPU device         : {gpus[0].name}')
else:
    print('   ⚠️  No GPU found! Go to Runtime → Change runtime type → T4 GPU')

---
## STEP 2 — Extract Dataset & Exploratory Data Analysis

In [ ]:
# ============================================================
# Extract the .7z dataset from Google Drive
# ============================================================

# Check if already extracted (saves time on re-runs)
if os.path.exists(LOCAL_DATA_DIR):
    print(f'✅ Dataset already extracted at {LOCAL_DATA_DIR}')
else:
    print('📦 Extracting dataset...')
    !7z x "{DRIVE_DATA_PATH}" -o/content/ -y
    print(f'✅ Dataset extracted to {LOCAL_DATA_DIR}')

# Verify folders exist
folders = [f for f in os.listdir(LOCAL_DATA_DIR)
           if os.path.isdir(os.path.join(LOCAL_DATA_DIR, f))]
print(f'\nFolders found: {sorted(folders)}')

In [ ]:
# ============================================================
# Count images per class
# ============================================================

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def count_images(data_dir):
    """Count images per class folder."""
    counts = {}
    for class_name in sorted(os.listdir(data_dir)):
        class_path = os.path.join(data_dir, class_name)
        if os.path.isdir(class_path):
            n = sum(1 for f in os.listdir(class_path)
                    if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS)
            counts[class_name] = n
    return counts

class_counts = count_images(LOCAL_DATA_DIR)
total_images = sum(class_counts.values())

df_counts = pd.DataFrame(
    list(class_counts.items()),
    columns=['Disease Class', 'Image Count']
)
df_counts['Percentage'] = (df_counts['Image Count'] / total_images * 100).round(1)

print('='*40)
print('DATASET SUMMARY')
print('='*40)
print(df_counts.to_string(index=False))
print(f'{'='*40}')
print(f'TOTAL: {total_images} images')

In [ ]:
# ============================================================
# Visualize class distribution
# ============================================================

PALETTE = sns.color_palette('Set2', len(class_counts))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset Class Distribution', fontsize=15, fontweight='bold')

# Bar chart
names  = list(class_counts.keys())
counts = list(class_counts.values())
bars   = ax1.bar(names, counts, color=PALETTE)
ax1.set_xlabel('Disease Class')
ax1.set_ylabel('Number of Images')
ax1.set_title('Image Count per Class')
ax1.grid(axis='y', alpha=0.3)
for bar, count in zip(bars, counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(count), ha='center', va='bottom', fontweight='bold')

# Pie chart
ax2.pie(counts, labels=names, colors=PALETTE,
        autopct='%1.1f%%', startangle=140)
ax2.set_title('Class Distribution (%)')

plt.tight_layout()
plt.savefig(f'{PLOTS_SAVE_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Class distribution plot saved')

In [ ]:
# ============================================================
# Show sample images from each class
# ============================================================

fig, axes = plt.subplots(4, 5, figsize=(16, 13))
fig.suptitle('Sample Images from Each Disease Class', fontsize=15, fontweight='bold')

for row_idx, class_name in enumerate(sorted(class_counts.keys())):
    class_path = os.path.join(LOCAL_DATA_DIR, class_name)
    images     = [f for f in os.listdir(class_path)
                  if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS]
    sample     = random.sample(images, min(5, len(images)))

    for col_idx, img_file in enumerate(sample):
        img  = load_img(os.path.join(class_path, img_file), target_size=(224, 224))
        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx][col_idx].set_title(class_name, fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(f'{PLOTS_SAVE_DIR}/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sample images plot saved')

---
## STEP 3 — Remove Duplicate Images

In [ ]:
# ============================================================
# Detect and remove exact duplicate images using MD5 hash
#
# Why MD5?
#   - Fast checksum of file bytes
#   - Two identical images always produce the same hash
#   - Different images almost never produce the same hash
# ============================================================

def compute_md5(file_path):
    """Compute MD5 hash of a file."""
    with open(file_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()


def remove_duplicates(data_dir):
    """Remove exact duplicate images from the dataset."""
    seen_hashes        = {}       # hash → first file path
    duplicates_removed = {}       # class_name → count removed
    total_removed      = 0

    for class_name in sorted(os.listdir(data_dir)):
        class_path = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_path):
            continue

        duplicates_removed[class_name] = 0

        for img_file in os.listdir(class_path):
            img_path = os.path.join(class_path, img_file)
            if not os.path.isfile(img_path):
                continue

            file_hash = compute_md5(img_path)

            if file_hash in seen_hashes:
                os.remove(img_path)              # Remove the duplicate
                duplicates_removed[class_name] += 1
                total_removed += 1
            else:
                seen_hashes[file_hash] = img_path

    return duplicates_removed, total_removed


print('🔍 Scanning for duplicate images...')
dup_counts, total_removed = remove_duplicates(LOCAL_DATA_DIR)

print('\nDuplicate Removal Results:')
print('-' * 35)
for cls, count in dup_counts.items():
    print(f'  {cls:<20} : {count} duplicates removed')
print(f'  {"TOTAL":<20} : {total_removed} duplicates removed')
print()

# Recount after removal
class_counts_clean = count_images(LOCAL_DATA_DIR)
total_clean        = sum(class_counts_clean.values())
print(f'✅ Clean dataset: {total_clean} images remaining')

---
## STEP 4 — Split Dataset into Train / Validation / Test

In [ ]:
# ============================================================
# Stratified Train / Val / Test Split
#
# Ratios: 60% Train | 20% Validation | 20% Test
#
# Stratified = each split has the SAME class proportions
# This prevents data imbalance between splits
# ============================================================

TRAIN_RATIO = 0.60
VAL_RATIO   = 0.20
TEST_RATIO  = 0.20


def build_file_list(data_dir):
    """Collect all image paths and their class labels."""
    file_paths, labels = [], []
    for class_name in sorted(os.listdir(data_dir)):
        class_path = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_path):
            continue
        for img_file in os.listdir(class_path):
            if os.path.splitext(img_file)[1].lower() in VALID_EXTENSIONS:
                file_paths.append(os.path.join(class_path, img_file))
                labels.append(class_name)
    return file_paths, labels


def copy_images_to_split(paths, labels, target_dir):
    """Copy images into class subfolders in target_dir."""
    for src_path, label in zip(paths, labels):
        dst_dir  = os.path.join(target_dir, label)
        os.makedirs(dst_dir, exist_ok=True)
        dst_path = os.path.join(dst_dir, os.path.basename(src_path))
        if not os.path.exists(dst_path):
            shutil.copy2(src_path, dst_path)


# Check if split already done (avoids re-copying on re-run)
split_done = all(
    os.path.exists(os.path.join(LOCAL_SPLIT_DIR, s))
    for s in ['train', 'val', 'test']
)

if split_done:
    print('✅ Split already exists, loading from disk...')
else:
    print('📂 Building train/val/test splits...')
    all_paths, all_labels = build_file_list(LOCAL_DATA_DIR)

    # Split 1: Separate test set (20%)
    train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
        all_paths, all_labels,
        test_size=TEST_RATIO,
        stratify=all_labels,
        random_state=RANDOM_SEED
    )

    # Split 2: Separate val from remaining (25% of 80% = 20% overall)
    val_fraction = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        train_val_paths, train_val_labels,
        test_size=val_fraction,
        stratify=train_val_labels,
        random_state=RANDOM_SEED
    )

    # Copy to split directories
    copy_images_to_split(train_paths, train_labels, f'{LOCAL_SPLIT_DIR}/train')
    copy_images_to_split(val_paths,   val_labels,   f'{LOCAL_SPLIT_DIR}/val')
    copy_images_to_split(test_paths,  test_labels,  f'{LOCAL_SPLIT_DIR}/test')
    print('✅ Splits created')

# Count images in each split
split_counts = {}
for split in ['train', 'val', 'test']:
    split_path = os.path.join(LOCAL_SPLIT_DIR, split)
    total = sum(
        len([f for f in os.listdir(os.path.join(split_path, cls))
             if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS])
        for cls in os.listdir(split_path)
        if os.path.isdir(os.path.join(split_path, cls))
    )
    split_counts[split] = total

print('\nSplit Summary:')
print('-' * 30)
for split, count in split_counts.items():
    pct = count / sum(split_counts.values()) * 100
    print(f'  {split:<8}: {count:>5} images ({pct:.0f}%)')

# Store for later use
train_val_paths_all  = []
train_val_labels_all = []
for split in ['train', 'val']:
    p, l = build_file_list(os.path.join(LOCAL_SPLIT_DIR, split))
    train_val_paths_all  += p
    train_val_labels_all += l

---
## STEP 5 — Data Generators with Augmentation

In [ ]:
# ============================================================
# Create Keras ImageDataGenerators
#
# Training:         WITH augmentation (creates variety)
# Validation/Test:  NO augmentation   (fair evaluation)
# ============================================================

# Training generator — augmented
train_datagen = ImageDataGenerator(
    rescale=1.0/255,            # Normalize pixels to [0, 1]
    rotation_range=40,          # Random rotation ±40°
    width_shift_range=0.2,      # Random horizontal shift
    height_shift_range=0.2,     # Random vertical shift
    shear_range=0.2,            # Shear transformation
    zoom_range=0.2,             # Random zoom in/out
    horizontal_flip=True,       # Mirror horizontally
    brightness_range=[0.5, 1.5],# Vary brightness
    fill_mode='nearest'         # Fill gaps after transforms
)

# Validation / Test generator — only rescaling
eval_datagen = ImageDataGenerator(rescale=1.0/255)


def make_generator(datagen, split_name, shuffle=False):
    """Helper to create a generator from a split directory."""
    return datagen.flow_from_directory(
        directory=os.path.join(LOCAL_SPLIT_DIR, split_name),
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=shuffle,
        seed=RANDOM_SEED
    )


train_generator = make_generator(train_datagen, 'train', shuffle=True)
val_generator   = make_generator(eval_datagen,  'val',   shuffle=False)
test_generator  = make_generator(eval_datagen,  'test',  shuffle=False)

CLASS_INDICES = train_generator.class_indices
print(f'\nClass indices: {CLASS_INDICES}')
print(f'Train samples  : {train_generator.samples}')
print(f'Val samples    : {val_generator.samples}')
print(f'Test samples   : {test_generator.samples}')

In [ ]:
# ============================================================
# Visualize augmented images
# (So we can see what augmentation looks like)
# ============================================================

# Pick one sample image for augmentation demo
sample_class = list(class_counts.keys())[0]
sample_dir   = os.path.join(LOCAL_DATA_DIR, sample_class)
sample_img   = os.path.join(sample_dir, os.listdir(sample_dir)[0])

img_array = img_to_array(load_img(sample_img, target_size=IMAGE_SIZE))
img_array = img_array.reshape((1,) + img_array.shape)

aug_gen = train_datagen.flow(img_array, batch_size=1)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle(f'Data Augmentation Examples — {sample_class}',
             fontsize=13, fontweight='bold')

# Original
axes[0][0].imshow(load_img(sample_img, target_size=IMAGE_SIZE))
axes[0][0].set_title('Original', fontweight='bold')
axes[0][0].axis('off')

# Augmented versions
for i, ax in enumerate(axes.flat[1:]):
    aug_img = next(aug_gen)[0].astype('uint8')
    ax.imshow(aug_img)
    ax.set_title(f'Augmented {i+1}')
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{PLOTS_SAVE_DIR}/augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Augmentation examples saved')

---
## STEP 6 — Build CNN Models with Transfer Learning

In [ ]:
# ============================================================
# Model builder — same architecture pattern for all 4 models
#
# Design:
#   1. Pre-trained base (ImageNet) → frozen initially
#   2. GlobalAveragePooling2D      → reduces to 1D vector
#   3. Dense(1024, relu)           → task-specific features
#   4. Dropout(0.5)                → prevents overfitting
#   5. Dense(4, softmax)           → disease probabilities
#   6. Unfreeze last N layers      → fine-tuning
# ============================================================

MODEL_CONFIGS = {
    'VGG16'      : {'class': VGG16,       'unfreeze': 4,  'input': (224, 224, 3)},
    'ResNet50'   : {'class': ResNet50,    'unfreeze': 10, 'input': (224, 224, 3)},
    'InceptionV3': {'class': InceptionV3, 'unfreeze': 10, 'input': (224, 224, 3)},
    'Xception'   : {'class': Xception,    'unfreeze': 10, 'input': (224, 224, 3)},
}


def build_model(model_name, num_classes=NUM_CLASSES,
                dense_units=1024, dropout_rate=0.5,
                learning_rate=LEARNING_RATE):
    """
    Build a transfer learning model.

    Args:
        model_name   : One of MODEL_CONFIGS keys
        num_classes  : Number of output disease classes
        dense_units  : Neurons in the custom dense layer
        dropout_rate : Dropout probability (0 to 1)
        learning_rate: Adam optimizer learning rate

    Returns:
        Compiled Keras model
    """
    cfg         = MODEL_CONFIGS[model_name]
    input_shape = cfg['input']
    unfreeze_n  = cfg['unfreeze']

    # 1. Load pre-trained base (no top classification layer)
    base = cfg['class'](
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # 2. Freeze ALL base layers first
    for layer in base.layers:
        layer.trainable = False

    # 3. Add custom classification head
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(dense_units, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=output)

    # 4. Unfreeze last N layers for fine-tuning
    for layer in base.layers[-unfreeze_n:]:
        layer.trainable = True

    # 5. Compile
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )

    total     = len(model.layers)
    trainable = sum(1 for l in model.layers if l.trainable)
    print(f'  {model_name}: {total} layers total, {trainable} trainable')

    return model


def get_callbacks(model_name, fold_num=None):
    """Standard training callbacks."""
    suffix   = f'_fold{fold_num}' if fold_num else ''
    filepath = f'{MODEL_SAVE_DIR}/best_{model_name}{suffix}.keras'
    return [
        EarlyStopping(
            monitor='val_loss', patience=10,
            restore_best_weights=True, verbose=0
        ),
        ModelCheckpoint(
            filepath=filepath, monitor='val_loss',
            save_best_only=True, verbose=0
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.2, patience=5,
            min_lr=1e-6, verbose=0
        ),
    ]


# Quick test — build VGG16 and print summary
print('Building models (test):')
test_model = build_model('VGG16')
print(f'\nVGG16 output shape: {test_model.output_shape}')
del test_model
tf.keras.backend.clear_session()
print('✅ Model builder working correctly')

---
## STEP 7 — K-Fold Cross Validation

**Why K-Fold?**  
A single train/val split can be lucky or unlucky. With 5-Fold CV, every image is used for validation exactly once. The mean and std of accuracy across folds gives a reliable, honest estimate of model performance — required for a rigorous Master's thesis.

In [ ]:
# ============================================================
# Stratified K-Fold Cross Validation
#
# Runs for ONE model at a time to save memory.
# Change MODEL_TO_CV to run for different models.
# ============================================================

# Change this to run CV for a different model
MODEL_TO_CV = 'VGG16'   # Options: 'VGG16', 'ResNet50', 'InceptionV3', 'Xception'

print(f'Running {N_FOLDS}-Fold Cross Validation for {MODEL_TO_CV}')
print('='*55)

all_paths_cv  = np.array(train_val_paths_all)
all_labels_cv = np.array(train_val_labels_all)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

cv_results = []
cv_fold_dir = f'/content/cv_folds/{MODEL_TO_CV}'

for fold, (train_idx, val_idx) in enumerate(
    skf.split(all_paths_cv, all_labels_cv), start=1
):
    print(f'\n--- Fold {fold}/{N_FOLDS} ---')

    fold_train_paths  = all_paths_cv[train_idx].tolist()
    fold_train_labels = all_labels_cv[train_idx].tolist()
    fold_val_paths    = all_paths_cv[val_idx].tolist()
    fold_val_labels   = all_labels_cv[val_idx].tolist()

    # Copy fold images to temp directories
    fold_train_dir = f'{cv_fold_dir}/fold_{fold}/train'
    fold_val_dir   = f'{cv_fold_dir}/fold_{fold}/val'
    copy_images_to_split(fold_train_paths, fold_train_labels, fold_train_dir)
    copy_images_to_split(fold_val_paths,   fold_val_labels,   fold_val_dir)

    # Create generators for this fold
    fold_train_gen = train_datagen.flow_from_directory(
        fold_train_dir, target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical',
        shuffle=True, seed=RANDOM_SEED
    )
    fold_val_gen = eval_datagen.flow_from_directory(
        fold_val_dir, target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical',
        shuffle=False
    )

    # Compute class weights for this fold
    fold_labels   = fold_train_gen.classes
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(fold_labels),
        y=fold_labels
    )
    class_weights_dict = dict(enumerate(class_weights))

    # Build and train model
    model = build_model(MODEL_TO_CV)
    history = model.fit(
        fold_train_gen,
        epochs=EPOCHS,
        validation_data=fold_val_gen,
        class_weight=class_weights_dict,
        callbacks=get_callbacks(MODEL_TO_CV, fold_num=fold),
        verbose=1
    )

    # Evaluate on validation set
    results = model.evaluate(fold_val_gen, verbose=0)
    metric_names = model.metrics_names
    metrics_dict = dict(zip(metric_names, results))

    cv_results.append({
        'fold'         : fold,
        'val_loss'     : metrics_dict.get('loss'),
        'val_accuracy' : metrics_dict.get('accuracy'),
        'val_precision': metrics_dict.get('precision'),
        'val_recall'   : metrics_dict.get('recall'),
    })

    print(f'  Fold {fold} → Loss: {metrics_dict["loss"]:.4f} | '
          f'Accuracy: {metrics_dict["accuracy"]:.4f} | '
          f'Precision: {metrics_dict["precision"]:.4f} | '
          f'Recall: {metrics_dict["recall"]:.4f}')

    # Free memory before next fold
    del model
    tf.keras.backend.clear_session()

# Save CV results
cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(f'{REPORTS_SAVE_DIR}/cv_results_{MODEL_TO_CV}.csv', index=False)

# Print summary
print(f'\n{"="*55}')
print(f'CV SUMMARY — {MODEL_TO_CV}')
print(f'{"="*55}')
print(cv_df.round(4).to_string(index=False))
print(f'\nMean Accuracy : {cv_df["val_accuracy"].mean():.4f}')
print(f'Std  Accuracy : {cv_df["val_accuracy"].std():.4f}')
print(f'Min  Accuracy : {cv_df["val_accuracy"].min():.4f}')
print(f'Max  Accuracy : {cv_df["val_accuracy"].max():.4f}')

# Store for statistical tests later
all_cv_results = {MODEL_TO_CV: cv_results}

---
## STEP 8 — Train Final Models on Full Training Data

In [ ]:
# ============================================================
# Train FINAL models on complete train set
#
# After CV confirms model quality, we train one final model
# on ALL train+val data for maximum performance.
# This is the model we save and deploy.
#
# Set MODELS_TO_TRAIN to control which models run.
# ============================================================

# Change this list to train specific models
# All 4: ['VGG16', 'ResNet50', 'InceptionV3', 'Xception']
# Just VGG16: ['VGG16']
MODELS_TO_TRAIN = ['VGG16', 'ResNet50', 'InceptionV3', 'Xception']

all_histories       = {}
all_final_metrics   = {}
all_predictions     = {}   # Stores y_pred per model for statistical tests

# Compute class weights once
class_weights_final = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights_dict  = dict(enumerate(class_weights_final))

print('Class weights (handles imbalance):')
for idx, (cls, w) in enumerate(zip(
    list(train_generator.class_indices.keys()), class_weights_final
)):
    print(f'  {cls}: {w:.4f}')

print(f'\nTraining models: {MODELS_TO_TRAIN}')
print('This may take 20-40 minutes per model on T4 GPU\n')


for model_name in MODELS_TO_TRAIN:
    print(f'\n{"="*55}')
    print(f'Training FINAL model: {model_name}')
    print(f'{"="*55}')

    model = build_model(model_name)

    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=val_generator,
        class_weight=class_weights_dict,
        callbacks=get_callbacks(model_name),
        verbose=1
    )

    # Save history and model
    all_histories[model_name] = history.history

    model_path = f'{MODEL_SAVE_DIR}/final_{model_name}.keras'
    model.save(model_path)

    # Save history to JSON
    hist_path = f'{REPORTS_SAVE_DIR}/history_{model_name}.json'
    with open(hist_path, 'w') as f:
        json.dump(
            {k: [float(x) for x in v] for k, v in history.history.items()}, f
        )

    # Evaluate on test set
    test_generator.reset()
    y_true       = test_generator.classes
    y_pred_proba = model.predict(test_generator, verbose=0)
    y_pred       = np.argmax(y_pred_proba, axis=1)

    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall    = recall_score(y_true, y_pred,    average='macro', zero_division=0)
    f1        = f1_score(y_true, y_pred,        average='macro', zero_division=0)
    auc_score = roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='macro')

    all_final_metrics[model_name] = {
        'accuracy' : accuracy,
        'precision': precision,
        'recall'   : recall,
        'f1'       : f1,
        'auc'      : auc_score,
    }
    all_predictions[model_name] = {
        'y_true'      : y_true.tolist(),
        'y_pred'      : y_pred.tolist(),
        'y_pred_proba': y_pred_proba.tolist(),
    }

    print(f'\n  ✅ {model_name} Final Test Results:')
    print(f'     Accuracy  : {accuracy:.4f}')
    print(f'     Precision : {precision:.4f}')
    print(f'     Recall    : {recall:.4f}')
    print(f'     F1-Score  : {f1:.4f}')
    print(f'     AUC-ROC   : {auc_score:.4f}')
    print(f'     Saved to  : {model_path}')

    del model
    tf.keras.backend.clear_session()

print('\n✅ All models trained and evaluated!')

---
## STEP 9 — Results Visualization

In [ ]:
# ============================================================
# Training curves for each model
# ============================================================

def plot_training_curves(history_dict, model_name):
    """Plot loss, accuracy, precision and recall over epochs."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Training History — {model_name}',
                 fontsize=15, fontweight='bold')

    metrics = [
        ('loss',      'val_loss',      'Loss'),
        ('accuracy',  'val_accuracy',  'Accuracy'),
        ('precision', 'val_precision', 'Precision'),
        ('recall',    'val_recall',    'Recall'),
    ]

    for ax, (tr_key, val_key, title) in zip(axes.flat, metrics):
        if tr_key not in history_dict:
            ax.set_visible(False)
            continue

        epochs = range(1, len(history_dict[tr_key]) + 1)
        ax.plot(epochs, history_dict[tr_key],  label='Train',      color='royalblue', lw=2)
        ax.plot(epochs, history_dict[val_key], label='Validation', color='tomato',    lw=2, ls='--')

        if 'loss' in tr_key:
            best = int(np.argmin(history_dict[val_key]))
            best_val = min(history_dict[val_key])
        else:
            best = int(np.argmax(history_dict[val_key]))
            best_val = max(history_dict[val_key])

        ax.axvline(best + 1, color='green', ls=':', alpha=0.7,
                   label=f'Best epoch={best+1}')
        ax.scatter([best + 1], [best_val], color='green', s=80, zorder=5)
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(title)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{PLOTS_SAVE_DIR}/training_curves_{model_name}.png',
                dpi=150, bbox_inches='tight')
    plt.show()


for model_name, history in all_histories.items():
    plot_training_curves(history, model_name)
    print(f'✅ Training curves saved for {model_name}')

In [ ]:
# ============================================================
# Confusion Matrix for each model
# ============================================================

def plot_confusion_matrix(y_true, y_pred, class_names, model_name):
    """Plot normalized and raw confusion matrix side by side."""
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Confusion Matrix — {model_name}',
                 fontsize=14, fontweight='bold')

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax1)
    ax1.set_title('Raw Counts')
    ax1.set_xlabel('Predicted')
    ax1.set_ylabel('True')

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax2, vmin=0, vmax=1)
    ax2.set_title('Normalized (Row %)')
    ax2.set_xlabel('Predicted')
    ax2.set_ylabel('True')

    plt.tight_layout()
    plt.savefig(f'{PLOTS_SAVE_DIR}/confusion_matrix_{model_name}.png',
                dpi=150, bbox_inches='tight')
    plt.show()


class_name_list = list(train_generator.class_indices.keys())

for model_name, preds in all_predictions.items():
    plot_confusion_matrix(
        preds['y_true'], preds['y_pred'],
        class_name_list, model_name
    )
    print(f'✅ Confusion matrix saved for {model_name}')

In [ ]:
# ============================================================
# ROC-AUC Curves (One-vs-Rest for each class)
# ============================================================

def plot_roc_curves(y_true, y_pred_proba, class_names, model_name):
    """Plot per-class ROC curves with AUC scores."""
    n_classes  = len(class_names)
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
    palette    = sns.color_palette('Set1', n_classes)

    fig, ax = plt.subplots(figsize=(9, 7))

    for i, cls_name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc_val = sk_auc(fpr, tpr)
        ax.plot(fpr, tpr, color=palette[i], lw=2,
                label=f'{cls_name} (AUC = {roc_auc_val:.4f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.02])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves (One-vs-Rest) — {model_name}')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{PLOTS_SAVE_DIR}/roc_curves_{model_name}.png',
                dpi=150, bbox_inches='tight')
    plt.show()


for model_name, preds in all_predictions.items():
    plot_roc_curves(
        preds['y_true'],
        np.array(preds['y_pred_proba']),
        class_name_list, model_name
    )
    print(f'✅ ROC curves saved for {model_name}')

In [ ]:
# ============================================================
# Model Comparison Table & Bar Chart
# ============================================================

# Build comparison DataFrame
comparison_df = pd.DataFrame(all_final_metrics).T.round(4)
comparison_df.index.name = 'Model'
comparison_df.columns    = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']

print('='*65)
print('MODEL PERFORMANCE COMPARISON ON TEST SET')
print('='*65)
print(comparison_df.to_string())
comparison_df.to_csv(f'{REPORTS_SAVE_DIR}/model_comparison.csv')

# Highlight best model in each metric
print('\nBest model per metric:')
for col in comparison_df.columns:
    best_model = comparison_df[col].idxmax()
    best_val   = comparison_df[col].max()
    print(f'  {col:<12}: {best_model} ({best_val:.4f})')

# Bar chart
metrics  = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
palette  = sns.color_palette('Set2', len(comparison_df))

fig, ax = plt.subplots(figsize=(13, 6))
x       = np.arange(len(metrics))
width   = 0.8 / len(comparison_df)

for i, (model_name, row) in enumerate(comparison_df.iterrows()):
    offset = (i - len(comparison_df)/2 + 0.5) * width
    values = [row[m] for m in metrics]
    bars   = ax.bar(x + offset, values, width, label=model_name, color=palette[i])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison on Test Set', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim([max(0, comparison_df.values.min() - 0.05), 1.02])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_SAVE_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparison chart saved')

---
## STEP 10 — Statistical Tests

This is what separates a **Master's thesis** from a regular project.  
We prove statistically that VGG16 is significantly better — not just by chance.

In [ ]:
# ============================================================
# McNemar's Test — Are two models making different errors?
#
# Tests whether the DIFFERENCE in error rates between two
# models is statistically significant.
#
# p-value < 0.05 → models are significantly different
# p-value ≥ 0.05 → no significant difference
# ============================================================

from itertools import combinations


def mcnemar_test(y_true, pred_a, pred_b):
    """McNemar's test with continuity correction."""
    correct_a = (np.array(pred_a) == np.array(y_true))
    correct_b = (np.array(pred_b) == np.array(y_true))

    n01 = np.sum(~correct_a & correct_b)  # A wrong, B right
    n10 = np.sum(correct_a & ~correct_b)  # A right,  B wrong

    if (n01 + n10) == 0:
        return {'statistic': 0.0, 'p_value': 1.0, 'significant': False,
                'n01': 0, 'n10': 0}

    statistic = (abs(n01 - n10) - 1)**2 / (n01 + n10)
    p_value   = stats.chi2.sf(statistic, df=1)

    return {
        'statistic'  : round(float(statistic), 4),
        'p_value'    : round(float(p_value), 6),
        'significant': bool(p_value < 0.05),
        'n01'        : int(n01),
        'n10'        : int(n10),
    }


def wilcoxon_test(scores_a, scores_b):
    """Wilcoxon signed-rank test on CV fold scores."""
    try:
        stat, p = stats.wilcoxon(scores_a, scores_b)
        return {'statistic': round(float(stat), 4),
                'p_value'  : round(float(p), 6),
                'significant': bool(p < 0.05)}
    except ValueError:
        return {'statistic': 0.0, 'p_value': 1.0, 'significant': False}


def paired_ttest(scores_a, scores_b):
    """Paired t-test on CV fold scores."""
    stat, p = stats.ttest_rel(scores_a, scores_b)
    return {'statistic': round(float(stat), 4),
            'p_value'  : round(float(p), 6),
            'significant': bool(p < 0.05)}


# Run McNemar's test for every model pair
model_names   = list(all_predictions.keys())
y_true_shared = all_predictions[model_names[0]]['y_true']
stat_rows     = []

print('McNemar\'s Test Results')
print('='*65)
print(f'{"Model A":<14} {"Model B":<14} {"Statistic":>10} {"p-value":>10} {"Significant":>12}')
print('-'*65)

for model_a, model_b in combinations(model_names, 2):
    pred_a = all_predictions[model_a]['y_pred']
    pred_b = all_predictions[model_b]['y_pred']

    result = mcnemar_test(y_true_shared, pred_a, pred_b)
    sig    = '✅ YES' if result['significant'] else '❌ NO'

    print(f'{model_a:<14} {model_b:<14} {result["statistic"]:>10.4f} '
          f'{result["p_value"]:>10.6f} {sig:>12}')

    stat_rows.append({
        'Model A'            : model_a,
        'Model B'            : model_b,
        'McNemar Statistic'  : result['statistic'],
        'McNemar p-value'    : result['p_value'],
        'Significant (p<0.05)': result['significant'],
        'A wrong B right (n01)': result['n01'],
        'A right B wrong (n10)': result['n10'],
    })

stat_df = pd.DataFrame(stat_rows)
stat_df.to_csv(f'{REPORTS_SAVE_DIR}/statistical_tests.csv', index=False)
print('\n✅ Statistical test results saved')

In [ ]:
# ============================================================
# Bootstrap Confidence Intervals for each model's accuracy
# ============================================================

def bootstrap_ci(y_true, y_pred, n_iter=1000, confidence=0.95):
    """95% confidence interval via bootstrap resampling."""
    rng    = np.random.RandomState(RANDOM_SEED)
    n      = len(y_true)
    scores = []
    for _ in range(n_iter):
        idx = rng.randint(0, n, n)
        scores.append(accuracy_score(
            np.array(y_true)[idx],
            np.array(y_pred)[idx]
        ))
    alpha = 1 - confidence
    return (
        round(np.percentile(scores, 100 * alpha / 2), 4),
        round(np.percentile(scores, 100 * (1 - alpha / 2)), 4)
    )


print('95% Bootstrap Confidence Intervals — Accuracy')
print('='*55)
print(f'{"Model":<15} {"Accuracy":>10} {"95% CI Lower":>14} {"95% CI Upper":>14}')
print('-'*55)

for model_name, preds in all_predictions.items():
    acc  = all_final_metrics[model_name]['accuracy']
    lo, hi = bootstrap_ci(preds['y_true'], preds['y_pred'])
    print(f'{model_name:<15} {acc:>10.4f} {lo:>14.4f} {hi:>14.4f}')

print('\n✅ Confidence intervals computed')

---
## STEP 11 — Grad-CAM Explainability

**Why Grad-CAM?**  
Shows *which part of the leaf* the model looked at when making its decision.  
This is critical for thesis credibility — it proves the model learned actual disease features, not background noise.

In [ ]:
# ============================================================
# Grad-CAM implementation
# ============================================================

import cv2


def generate_gradcam(model, img_array, class_idx):
    """Generate Grad-CAM heatmap for a single image."""
    # Find last convolutional layer
    last_conv = None
    for layer in reversed(model.layers):
        if len(layer.output_shape) == 4:
            last_conv = layer.name
            break

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        loss = preds[:, class_idx]

    grads       = tape.gradient(loss, conv_out)
    pooled      = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap     = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap     = tf.squeeze(heatmap)
    heatmap     = tf.maximum(heatmap, 0)
    heatmap     = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def show_gradcam(model, img_path, class_names, model_name):
    """Load image, predict, generate and show Grad-CAM overlay."""
    img       = load_img(img_path, target_size=IMAGE_SIZE)
    img_array = img_to_array(img) / 255.0
    batch     = np.expand_dims(img_array, axis=0)

    preds     = model.predict(batch, verbose=0)[0]
    class_idx = int(np.argmax(preds))
    conf      = float(preds[class_idx])

    heatmap = generate_gradcam(model, batch, class_idx)

    h, w    = IMAGE_SIZE
    hmap_r  = cv2.resize(heatmap, (w, h))
    hmap_c  = cv2.applyColorMap(np.uint8(255 * hmap_r), cv2.COLORMAP_JET)
    hmap_c  = cv2.cvtColor(hmap_c, cv2.COLOR_BGR2RGB)

    orig    = np.uint8(img_array * 255)
    overlay = cv2.addWeighted(orig, 0.6, hmap_c, 0.4, 0)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle(
        f'Grad-CAM — {model_name} | '
        f'Predicted: {class_names[class_idx]} ({conf:.1%})',
        fontsize=13, fontweight='bold'
    )

    axes[0].imshow(img)
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    axes[1].imshow(hmap_r, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap')
    axes[1].axis('off')

    axes[2].imshow(overlay)
    axes[2].set_title('Overlay on Leaf')
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig(
        f'{PLOTS_SAVE_DIR}/gradcam_{model_name}_{class_names[class_idx]}.png',
        dpi=150, bbox_inches='tight'
    )
    plt.show()


# Load the best VGG16 model and show Grad-CAM for one image per class
vgg16_model = tf.keras.models.load_model(
    f'{MODEL_SAVE_DIR}/final_VGG16.keras'
)

print('Generating Grad-CAM visualizations for VGG16...')
for class_name in sorted(class_counts.keys()):
    class_path = os.path.join(LOCAL_DATA_DIR, class_name)
    imgs       = [f for f in os.listdir(class_path)
                  if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS]
    sample_img = os.path.join(class_path, imgs[0])
    show_gradcam(vgg16_model, sample_img, class_name_list, 'VGG16')

del vgg16_model
tf.keras.backend.clear_session()
print('✅ Grad-CAM visualizations complete')

---
## STEP 12 — Save Everything to Google Drive

In [ ]:
# ============================================================
# Copy all outputs from Colab's local storage → Google Drive
#
# Why? Colab resets its local storage when the session ends.
# Everything saved to Drive is permanent.
# ============================================================

import shutil

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print(f'Copying outputs to Drive: {DRIVE_OUTPUT_DIR}')
print('-'*50)

# Copy models
for f in os.listdir(MODEL_SAVE_DIR):
    if f.endswith('.keras') and f.startswith('final_'):
        src = os.path.join(MODEL_SAVE_DIR, f)
        dst = os.path.join(DRIVE_OUTPUT_DIR, 'models', f)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / (1024**2)
        print(f'  ✅ Model : {f} ({size_mb:.1f} MB)')

# Copy plots
plots_drive = os.path.join(DRIVE_OUTPUT_DIR, 'plots')
if os.path.exists(PLOTS_SAVE_DIR):
    shutil.copytree(PLOTS_SAVE_DIR, plots_drive, dirs_exist_ok=True)
    n_plots = len(os.listdir(plots_drive))
    print(f'  ✅ Plots  : {n_plots} files copied')

# Copy reports
reports_drive = os.path.join(DRIVE_OUTPUT_DIR, 'reports')
if os.path.exists(REPORTS_SAVE_DIR):
    shutil.copytree(REPORTS_SAVE_DIR, reports_drive, dirs_exist_ok=True)
    n_reports = len(os.listdir(reports_drive))
    print(f'  ✅ Reports: {n_reports} files copied')

print('\n✅ All outputs saved to Google Drive!')
print(f'   Location: {DRIVE_OUTPUT_DIR}')
print('\nNext step: Download final_VGG16.keras and place it in:')
print('  rice_disease_project/outputs/models/vgg16/final_vgg16.keras')

In [ ]:
# ============================================================
# Final Summary
# ============================================================

print('\n' + '='*65)
print('TRAINING COMPLETE — FINAL SUMMARY')
print('='*65)
print(f'\nDataset    : {total_clean} images | {NUM_CLASSES} classes')
print(f'Split      : 60% train | 20% val | 20% test | {N_FOLDS}-fold CV')
print(f'\nFinal Test Set Results:')
print('-'*65)
print(f'{"Model":<15} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1":>10} {"AUC":>10}')
print('-'*65)

best_model  = None
best_acc    = 0
for model_name, m in all_final_metrics.items():
    star = ' ⭐' if m['accuracy'] == max(x['accuracy'] for x in all_final_metrics.values()) else ''
    print(f'{model_name:<15} {m["accuracy"]:>10.4f} {m["precision"]:>10.4f} '
          f'{m["recall"]:>10.4f} {m["f1"]:>10.4f} {m["auc"]:>10.4f}{star}')
    if m['accuracy'] > best_acc:
        best_acc   = m['accuracy']
        best_model = model_name

print('-'*65)
print(f'\n🏆 Best model: {best_model} (Accuracy: {best_acc:.4f})')
print(f'\nOutputs saved to: {DRIVE_OUTPUT_DIR}')
print('\nNow open VSCode and run: uvicorn app.main:app --reload')